In [1]:
from dotenv import load_dotenv
load_dotenv()

from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

True

In [2]:
# 1. SETUP STATE SCHEMA
class HyDEState(TypedDict):
    query: str               # The raw user question
    hypothetical_doc: str    # The fake LLM answer used for embedding
    documents: List[Document]# The real retrieved documents
    final_answer: str        # The final grounded output

# Mock Vector Store setup with real source material
embeddings = OpenAIEmbeddings()
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents([
    Document(page_content="Procedures governing the premature cancellation of leasehold estates require a written 30-day notice."),
    Document(page_content="Standard office infrastructure software requests must clear engineering approval.")
])

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

In [3]:
# ==========================================
# 2. DEFINE THE LANGGRAPH NODES
# ==========================================

def generate_hypothetical_node(state: HyDEState):
    """Step 1: Write a fake answer matching the text style of the source DB."""
    query = state["query"]
    
    hyde_prompt = ChatPromptTemplate.from_template(
        "Write a concise, formal paragraph answering the following question. "
        "Do not include introductory commentary, match the style of a legal or technical document.\n\n"
        "Question: {query}"
    )
    
    chain = hyde_prompt | llm
    response = chain.invoke({"query": query})
    return {"hypothetical_doc": response.content}


def retrieve_node(state: HyDEState):
    """Step 2: Embed the fake answer and search the real database."""
    # We pass the hypothetical text, NOT the user's raw short query
    fake_text = state["hypothetical_doc"]
    
    retrieved_docs = vectorstore.similarity_search(fake_text, k=1)
    return {"documents": retrieved_docs}


def generate_answer_node(state: HyDEState):
    """Step 3: Answer the user's actual question using the discovered real text."""
    query = state["query"]
    real_docs = state["documents"]
    
    context = "\n\n".join([doc.page_content for doc in real_docs])
    
    answer_prompt = ChatPromptTemplate.from_template(
        "You are a helpful assistant. Answer the user's question using ONLY the provided context.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}"
    )
    
    chain = answer_prompt | llm
    response = chain.invoke({"query": query, "context": context})
    return {"final_answer": response.content}

In [4]:
# ==========================================
# 3. COMPILE THE WORKFLOW
# ==========================================
workflow = StateGraph(HyDEState)

# Add nodes to graph
workflow.add_node("hyde_generator", generate_hypothetical_node)
workflow.add_node("retriever", retrieve_node)
workflow.add_node("answer_generator", generate_answer_node)

# Link nodes lineally
workflow.add_edge(START, "hyde_generator")
workflow.add_edge("hyde_generator", "retriever")
workflow.add_edge("retriever", "answer_generator")
workflow.add_edge("answer_generator", END)

app = workflow.compile()

In [5]:
# ==========================================
# 4. RUN THE SYSTEM
# ==========================================
# Short, abstract user query that lacks formal words like "leasehold estate"
user_question = "How do I stop a rental early?"

result = app.invoke({"query": user_question})

print("--- HYPOTHETICAL PARAGRAPH GENERATED BY LLM ---")
print(result["hypothetical_doc"])

print("\n--- REAL SOURCE DOCUMENT FOUND IN VECTORSTORE ---")
print(result["documents"][0].page_content)

print("\n--- FINAL SYSTEM ANSWER ---")
print(result["final_answer"])

--- HYPOTHETICAL PARAGRAPH GENERATED BY LLM ---
To terminate a rental agreement prior to its scheduled expiration, the tenant must provide written notice to the landlord in accordance with the terms specified in the lease, including any required notice period. It is essential to review the lease for specific clauses regarding early termination, which may outline conditions such as penalties or the necessity for the landlord to mitigate damages by re-renting the premises. If the lease includes an early termination clause, compliance with its stipulations is mandatory. Additionally, it is advisable to document all communications and agreements made between the tenant and landlord to ensure a clear record of the termination process.

--- REAL SOURCE DOCUMENT FOUND IN VECTORSTORE ---
Procedures governing the premature cancellation of leasehold estates require a written 30-day notice.

--- FINAL SYSTEM ANSWER ---
To stop a rental early, you need to provide a written 30-day notice.
